# 04 · Teammate race pace

Qualifying shows one-lap speed; this compares teammates' **race pace**.

**Clean laps:** only laps FastF1 marks as accurate. That flag already requires no pit in/out lap, a real lap time, sector times that add up, and green or yellow track status only, so safety car, VSC and red-flag laps are out.

**Wet races** are excluded for now: by default, races where anyone used intermediate or wet tyres (`tyres`). `tyres_or_rain` also excludes races with any rain recorded.

**Model**, fitted separately for each race:

`lap_time ~ driver + compound + compound × tyre_age + lap_number`

It adjusts each lap for the tyre compound, how worn that tyre is (each compound degrading at its own rate) and lap number (fuel burn-off and track grip). The teammate gap is the difference between the two drivers' estimated pace with all of that held equal. By default a robust (Huber) regression is used, so laps lost in traffic or to mistakes count less.

`GapSE` is the standard error of each race's gap: a race gap smaller than about twice its SE could be noise.

Run notebook 03 first.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Settings and model fit
WET_RULE = "tyres"  # @param ["tyres", "tyres_or_rain"]
MIN_LAPS = 10  # @param {type:"integer"}
ROBUST = True  # @param {type:"boolean"}

import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest_tracing import load_race_ti
from f1sim.race_pace import (wet_races, clean_dry_laps, teammate_race_gaps,
                             race_season_summary)

laps = load_race_ti("laps", get_paths())
wet = wet_races(laps, WET_RULE)
clean = clean_dry_laps(laps, exclude_wet=True, wet_rule=WET_RULE)
gaps, skipped = teammate_race_gaps(clean, min_laps=MIN_LAPS, robust=ROBUST)
summary = race_season_summary(gaps)

n_races = laps.groupby(["Year", "Round"]).ngroups
print(f"{len(clean):,} clean dry laps of {len(laps):,} | {n_races} races, "
      f"{len(wet)} excluded as wet, {len(skipped)} items left out (see table) | "
      f"{len(gaps) // 2} teammate comparisons | median gap SE {gaps['GapSE'].median():.3f}s")

In [ ]:
# @title Races excluded as wet
wet

In [ ]:
# @title Races or drivers the model left out (and why)
skipped

In [ ]:
# @title Season race-pace head-to-head
YEAR = 2025  # @param {type:"integer"}

summary.loc[summary["Year"] == YEAR,
            ["Team", "Driver", "Teammate", "Races", "H2H", "MedianGapSec", "MedianGapPct"]]

In [ ]:
# @title Median race-pace gap to teammate, by driver
season = summary[summary["Year"] == YEAR]
fig = px.bar(
    season.sort_values("MedianGapPct"),
    x="MedianGapPct", y="Driver", color="Team", orientation="h",
    hover_data=["Teammate", "H2H", "Races", "MedianGapSec"],
    labels={"MedianGapPct": "Median race-pace gap to teammate (%)"},
    title=f"{YEAR}: median race-pace gap to teammate (negative = faster)",
)
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(height=650, yaxis={"categoryorder": "total descending"})
fig.show()

In [ ]:
# @title One team, race by race (bars = ±2 standard errors)
TEAM = "McLaren"  # @param {type:"string"}

teams = sorted(gaps.loc[gaps["Year"] == YEAR, "Team"].unique())
if TEAM not in teams:
    print(f"'{TEAM}' not found in {YEAR}. Pick one of:", teams)
else:
    tg = gaps[(gaps["Year"] == YEAR) & (gaps["Team"] == TEAM)].copy()
    tg["Err"] = 2 * tg["GapSEPct"]
    fig = px.scatter(
        tg, x="Round", y="GapPct", color="Driver", error_y="Err",
        hover_data=["Event", "Teammate", "GapSec", "GapSE", "Laps", "TeammateLaps"],
        labels={"GapPct": "Race-pace gap to teammate (%)"},
        title=f"{TEAM} {YEAR}: race-pace gap by round",
    )
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [ ]:
# @title One driver across seasons
DRIVER = "ALO"  # @param {type:"string"}

d = summary[summary["Driver"] == DRIVER]
if d.empty:
    print(f"No data for '{DRIVER}'. Use the 3-letter code, e.g. HAM, VER, LEC.")
else:
    fig = px.line(
        d, x="Year", y="MedianGapPct", markers=True, text="Teammate",
        hover_data=["Team", "H2H", "Races"],
        labels={"MedianGapPct": "Median race-pace gap to teammate (%)"},
        title=f"{DRIVER}: median race-pace gap to teammate by season",
    )
    fig.update_traces(textposition="top center")
    fig.add_hline(y=0, line_dash="dot")
    fig.show()

In [ ]:
# @title Qualifying vs race pace, per driver (needs notebook 00's data)
from f1sim.ingest import load_quali
from f1sim.features import teammate_gaps, season_summary

quali = season_summary(teammate_gaps(load_quali(get_paths())))
both = summary.merge(
    quali[["Year", "Driver", "Teammate", "MedianGapPct"]],
    on=["Year", "Driver", "Teammate"], suffixes=("Race", "Quali"),
)
season = both[both["Year"] == YEAR]
fig = px.scatter(
    season, x="MedianGapPctQuali", y="MedianGapPctRace", text="Driver", color="Team",
    labels={"MedianGapPctQuali": "Quali gap to teammate (%)",
            "MedianGapPctRace": "Race-pace gap to teammate (%)"},
    title=f"{YEAR}: qualifying vs race pace (bottom-left = faster in both)",
)
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_dash="dot"); fig.add_vline(x=0, line_dash="dot")
fig.show()